In [ ]:
from typing import Callable, List, Tuple
import random 
import numpy as np
import pandas as pd

from geneticalgorithm2 import AlgorithmParams
from geneticalgorithm2 import GeneticAlgorithm2 as ga

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## Problem
---

In [ ]:
def rastrigin(x):
    """
    Rastrigin function is a non-convex 
    function used as a performance test 
    problem for optimization algorithms.

    It is a typical example of a non-linear
    multimodal function.
    """

    A = 10
    dim = len(x)
    return A * dim + sum([(xi**2 - A * np.cos(2 *np.pi * xi)) for xi in x])

## GA
---

In [ ]:
population_size = [100, 200, 300, 400]
mutation_probability = [0.001, 0.05, 0.01]
elit_ratio = [0, 0.1, 0.2, 0.25]
parents_portion = [0.25, 0.3, 0.35, 0.4]
crossover_type = ["one_point", "two_point", "uniform", "segment", "shuffle"]
mutation_type = ["uniform_by_x", "uniform_by_center", "gauss_by_center", "gauss_by_x"]
selection_type = ["fully_random", "roulette", "stochastic", "sigma_scaling", "ranking", "linear_ranking", "tournament"]

In [ ]:
EXPERIMENTS = 5
df = pd.DataFrame()
experiments = []
for i in range(0,EXPERIMENTS):
    algorithm_parameters = AlgorithmParams(
        max_num_iteration = None,
        max_iteration_without_improv = 100,
        population_size = random.choice(population_size),
        mutation_probability = random.choice(mutation_probability),
        elit_ratio = random.choice(elit_ratio),
        parents_portion = random.choice(parents_portion),
        crossover_type = random.choice(crossover_type),
        mutation_type = random.choice(mutation_type),
        selection_type = random.choice(selection_type),
    )

    dim = 3

    model = ga(
        dimension=dim,
        variable_type="real",
        variable_boundaries=np.array([(-5.12, 5.12)] * dim),
        algorithm_parameters=algorithm_parameters
    )

    model.run(function=rastrigin, no_plot=True)
    experiments.append(model.report)
    convergence_generation = len(model.report)  # number of generations until convergence (stopping for no improvement) 

    best_val_norm = 1 / (1 + abs(model.best_function))
    conv_gen_norm = 1 / (1 + convergence_generation)

    score = 1 - (best_val_norm + conv_gen_norm) / 2 # [0, 1], lower is better


    exp_dict = {
        "experiment": i+1,
        "population_size": algorithm_parameters.population_size,
        "mutation_probability": algorithm_parameters.mutation_probability,
        "elit_ratio": algorithm_parameters.elit_ratio,
        "parents_portion": algorithm_parameters.parents_portion,
        "crossover_type": algorithm_parameters.crossover_type,
        "mutation_type": algorithm_parameters.mutation_type,
        "selection_type": algorithm_parameters.selection_type,
        "convergence_generation": convergence_generation,
        "best_function_value": model.best_function,
        "score": score
    }

    df = pd.concat([df, pd.DataFrame([exp_dict])], ignore_index=True)

    plt.plot(model.report, label = f"Experiment {i+1}")


plt.xlabel('Generation')
plt.ylabel('Minimized function')
plt.title('Experiments Convergence')
plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
df.sort_values(by='score', ascending=False)

In [ ]:
def plot_function(f: Callable, dim: int, boundaries: Tuple[int, int, int, int], points: int = 100, colormap: str = "twilight", title: str = "Function Plot"):
    """
    Plots the given function f in 2D or 3D.

    Parameters:
    f : Callable
        The function to be plotted. It should take a list or array of length `dim` as input.
    dim : int
        The dimension of the input space. Should be either 2 or 3.
    boundaries : Tuple[int, int, int, int]
        The boundaries for the input variables -> xmin, xmax, ymin, ymax.
    colormap : str
        The colormap to be used for plotting.
    """

    xmin, xmax, ymin, ymax = boundaries
    x = np.linspace(xmin, xmax, points)
    y = np.linspace(ymin, ymax, points)
    X, Y = np.meshgrid(x, y)
    Z = np.array([[f([x_val, y_val]) for x_val in x] for y_val in y])
    
    if dim == 2:
        plt.figure(figsize=(10, 7))
        cp = plt.contourf(X, Y, Z, levels=50, cmap=colormap)
        plt.colorbar(cp)
        plt.title(title)
        plt.xlabel('X-axis')
        plt.ylabel('Y-axis')
        plt.show()

    elif dim == 3:
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(X, Y, Z, cmap=colormap)
        ax.set_title(title)
        ax.set_xlabel('X-axis')
        ax.set_ylabel('Y-axis')
        ax.set_zlabel('Z-axis')
        plt.show()

In [ ]:
plot_function(rastrigin, dim=3, boundaries=(-5.12, 5.12, -5.12, 5.12))